# Generate Table DDL Scripts

This notebook scans the mounted Lakehouse and generates CREATE TABLE DDL scripts for all existing tables.  
Captures: data types, NULL/NOT NULL constraints, DECIMAL precision/scale, and Delta table format.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, 
    DoubleType, FloatType, BooleanType, TimestampType, DateType,
    DecimalType, BinaryType, ShortType, ByteType, ArrayType, MapType
)
from notebookutils import mssparkutils
from datetime import datetime

## Helper Functions

In [ ]:
def map_spark_type_to_sql(spark_type, nullable=True):
    """
    Convert Spark DataType to SQL DDL string with proper formatting.
    Handles decimals with precision/scale, and all standard types.
    """
    # Show nullable status clearly - NULL as comment, NOT NULL as constraint
    nullable_str = " /* NULL */" if nullable else " NOT NULL"
    
    if isinstance(spark_type, DecimalType):
        return f"DECIMAL({spark_type.precision},{spark_type.scale}){nullable_str}"
    elif isinstance(spark_type, StringType):
        return f"STRING{nullable_str}"
    elif isinstance(spark_type, IntegerType):
        return f"INT{nullable_str}"
    elif isinstance(spark_type, LongType):
        return f"BIGINT{nullable_str}"
    elif isinstance(spark_type, DoubleType):
        return f"DOUBLE{nullable_str}"
    elif isinstance(spark_type, FloatType):
        return f"FLOAT{nullable_str}"
    elif isinstance(spark_type, BooleanType):
        return f"BOOLEAN{nullable_str}"
    elif isinstance(spark_type, TimestampType):
        return f"TIMESTAMP{nullable_str}"
    elif isinstance(spark_type, DateType):
        return f"DATE{nullable_str}"
    elif isinstance(spark_type, BinaryType):
        return f"BINARY{nullable_str}"
    elif isinstance(spark_type, ShortType):
        return f"SMALLINT{nullable_str}"
    elif isinstance(spark_type, ByteType):
        return f"TINYINT{nullable_str}"
    elif isinstance(spark_type, ArrayType):
        element_type = map_spark_type_to_sql(spark_type.elementType, spark_type.containsNull).replace(" NOT NULL", "").replace(" /* NULL */", "")
        return f"ARRAY<{element_type}>{nullable_str}"
    elif isinstance(spark_type, MapType):
        key_type = map_spark_type_to_sql(spark_type.keyType, False).replace(" NOT NULL", "")
        value_type = map_spark_type_to_sql(spark_type.valueType, spark_type.valueContainsNull).replace(" NOT NULL", "").replace(" /* NULL */", "")
        return f"MAP<{key_type},{value_type}>{nullable_str}"
    elif isinstance(spark_type, StructType):
        fields = ", ".join([
            f"{field.name}:{map_spark_type_to_sql(field.dataType, field.nullable).replace(' NOT NULL', '').replace(' /* NULL */', '')}"
            for field in spark_type.fields
        ])
        return f"STRUCT<{fields}>{nullable_str}"
    else:
        # Fallback for unknown types
        return f"{str(spark_type)}{nullable_str}"


def generate_create_table_ddl(table_name: str, schema: StructType, use_delta: bool = True) -> str:
    """
    Generate a CREATE TABLE DDL statement from a Spark schema.
    
    Args:
        table_name: Name of the table
        schema: Spark StructType schema
        use_delta: Whether to use DELTA format (default True for Fabric)
    
    Returns:
        Complete CREATE TABLE DDL string
    """
    columns = []
    for field in schema.fields:
        col_name = field.name
        col_type = map_spark_type_to_sql(field.dataType, field.nullable)
        columns.append(f"    {col_name} {col_type}")
    
    columns_ddl = ",\n".join(columns)
    using_clause = "USING DELTA" if use_delta else ""
    
    ddl = f"""CREATE TABLE IF NOT EXISTS {table_name} (
{columns_ddl}
)
{using_clause}"""
    
    return ddl

## Discover All Tables in Lakehouse

In [ ]:
# Get all tables from the default database (lakehouse)
tables = spark.catalog.listTables()

print(f"Found {len(tables)} tables in the lakehouse:\n")
for table in tables:
    print(f"  - {table.name} (database: {table.database})")

table_names = [table.name for table in tables]

## Generate DDL Scripts for All Tables

In [ ]:
ddl_scripts = []

print("="* 80)
print("GENERATING DDL SCRIPTS")
print("=" * 80)

for table_name in table_names:
    try:
        # Get table schema
        df = spark.table(table_name)
        schema = df.schema
        
        # Generate DDL
        ddl = generate_create_table_ddl(table_name, schema, use_delta=True)
        ddl_scripts.append({
            "table_name": table_name,
            "ddl": ddl
        })
        
        print(f"\n{'─' * 80}")
        print(f"Table: {table_name}")
        print(f"{'─' * 80}")
        print(ddl)
        
    except Exception as e:
        print(f"\n❌ ERROR generating DDL for table '{table_name}': {e}")

print(f"\n{'=' * 80}")
print(f"Generated {len(ddl_scripts)} DDL scripts successfully.")
print("=" * 80)

## Export DDL Scripts to File (Optional)

In [ ]:
# Generate a timestamp for the output file
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_filename = f"table_ddl_scripts_{timestamp}.sql"

# Combine all DDL scripts into one file
combined_ddl = f"""-- ============================================================================
-- Lakehouse Table DDL Scripts
-- Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
-- Total Tables: {len(ddl_scripts)}
-- ============================================================================

"""

for script in ddl_scripts:
    combined_ddl += f"""
-- ============================================================================
-- Table: {script['table_name']}
-- ============================================================================
{script['ddl']};

"""

# Write to lakehouse Files section
try:
    output_path = f"Files/{output_filename}"
    mssparkutils.fs.put(output_path, combined_ddl, overwrite=True)
    print(f"✅ DDL scripts exported to: {output_path}")
    print(f"   File size: {len(combined_ddl)} bytes")
except Exception as e:
    print(f"⚠️  Could not write to Files: {e}")
    print("Displaying content instead:\n")
    print(combined_ddl[:2000])  # Show first 2000 chars

## Summary: Schema Details by Table

In [ ]:
# Create a summary DataFrame showing table schemas
schema_details = []

for table_name in table_names:
    try:
        df = spark.table(table_name)
        for field in df.schema.fields:
            # Get detailed type info
            if isinstance(field.dataType, DecimalType):
                type_detail = f"DECIMAL({field.dataType.precision},{field.dataType.scale})"
            else:
                type_detail = str(field.dataType).replace("Type()", "").upper()
            
            schema_details.append({
                "table_name": table_name,
                "column_name": field.name,
                "data_type": type_detail,
                "nullable": "YES" if field.nullable else "NO"
            })
    except Exception as e:
        print(f"Could not read schema for {table_name}: {e}")

# Convert to DataFrame and display
if schema_details:
    from pyspark.sql import Row
    summary_df = spark.createDataFrame([Row(**d) for d in schema_details])
    print(f"\n📊 Schema Summary ({len(schema_details)} columns across {len(table_names)} tables):\n")
    summary_df.orderBy("table_name", "column_name").show(100, truncate=False)
else:
    print("No schema details to display.")